# Train, Evaluate & Export the Baseline LSTM

We train the `mindwave_stress_lstm` model on the cached WESAD wrist windows and:
1. report a **fast subject-wise hold-out** (S13 + S16) with rich plots,
2. run **Leave-One-Subject-Out** cross-validation for a trustworthy headline number,
3. fit a **final model on all subjects** and convert it to **TFLite** (float + int8) for the Android app

In [ ]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from src.config import LABEL_NAMES, MODELS_DIR, PROCESSED_DIR, set_global_seed
from src.train import (
    load_npz, run_holdout, run_loso, train_final_and_export,
)
from src.evaluate import (
    metrics_dict, plot_confusion_matrix, plot_training_curves, plot_roc_pr,
    compare_keras_vs_tflite,
)
from src.tflite_export import run_tflite

sns.set_theme(style='whitegrid')
set_global_seed()
print('TF:', tf.__version__)

## Load the cached dataset

In [ ]:
X, y, subject_ids, feature_names = load_npz(PROCESSED_DIR / 'wesad_wrist.npz')
print('X:', X.shape, 'y:', y.shape, '| subjects:', np.unique(subject_ids).tolist())

vc = pd.Series(y).map(LABEL_NAMES).value_counts()
fig, ax = plt.subplots(figsize=(5, 3))
vc.plot(kind='bar', color=['#9ad3bc', '#f76c5e', '#ffd166'], ax=ax)
ax.set_title('Window count per class'); ax.set_ylabel('# windows')
for c in ax.containers: ax.bar_label(c)

## Fast hold-out fold (subjects S13 + S16)

In [ ]:
model, scaler, history, metrics, y_score = run_holdout(
    X, y, subject_ids, holdout=('S13', 'S16'),
    n_classes=2, epochs=60, batch_size=64,
)
print(json.dumps(metrics, indent=2))

# Persist hold-out metrics
(MODELS_DIR / 'holdout_metrics.json').write_text(json.dumps(metrics, indent=2))
print('Saved holdout_metrics.json')

In [ ]:
plot_training_curves(history)
plt.suptitle('Hold-out training curves', y=1.02); plt.tight_layout()

In [ ]:
y_pred = y_score.argmax(1)
test_mask = np.isin(subject_ids, ['S13', 'S16'])
y_test = y[test_mask]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_confusion_matrix(y_test, y_pred, normalize=False, ax=axes[0], title='Counts')
plot_confusion_matrix(y_test, y_pred, normalize=True,  ax=axes[1], title='Row-normalised')
fig.tight_layout()

In [ ]:
per_cls = pd.DataFrame(metrics['per_class']).T
ax = per_cls[['precision', 'recall', 'f1']].plot.bar(figsize=(7, 3.5))
ax.set_ylim(0, 1); ax.set_title('Per-class precision / recall / F1')
ax.set_ylabel('score'); plt.xticks(rotation=0)

In [ ]:
plot_roc_pr(y_test, y_score)

## Leave-One-Subject-Out cross-validation

In [ ]:
loso_summary = run_loso(X, y, subject_ids, n_classes=2, epochs=40, batch_size=64)
print(json.dumps({k: v for k, v in loso_summary.items() if k != 'per_fold'}, indent=2))

# Persist LOSO metrics
(MODELS_DIR / 'loso_metrics.json').write_text(json.dumps(loso_summary, indent=2))
print('Saved loso_metrics.json')

fold_df = pd.DataFrame({
    sid: {'accuracy': m['accuracy'], 'macro_f1': m['macro_f1']}
    for sid, m in loso_summary['per_fold'].items()
}).T.sort_index()
display(fold_df.round(3))

fig, ax = plt.subplots(figsize=(7, 3.5))
fold_df.plot(kind='bar', ax=ax)
ax.set_ylim(0, 1); ax.axhline(loso_summary['mean_macro_f1'], ls='--', color='k', alpha=0.5)
ax.set_title('LOSO per-subject metrics'); ax.set_ylabel('score'); plt.xticks(rotation=45)

## 4. Train final model on all subjects + export to TFLite

In [ ]:
export_info = train_final_and_export(
    X, y, n_classes=3, epochs=40, batch_size=64,
    label_names=LABEL_NAMES, feature_names=feature_names,
)
export_info

## Keras vs TFLite numerical parity

In [ ]:
from tensorflow.keras.models import load_model
model = load_model(MODELS_DIR / 'mindwave_stress.keras')
sample = X[:128].astype(np.float32)
# Apply the same scaler used in train_final_and_export.
import joblib
scaler = joblib.load(MODELS_DIR / 'scaler.pkl')
n, t, f = sample.shape
sample_s = scaler.transform(sample.reshape(-1, f)).reshape(n, t, f).astype(np.float32)

keras_pred = model.predict(sample_s, verbose=0)
tfl_pred = run_tflite(MODELS_DIR / 'mindwave_stress.tflite', sample_s)
tfl_int8_pred = run_tflite(MODELS_DIR / 'mindwave_stress_int8.tflite', sample_s)
print('float TFLite parity :', compare_keras_vs_tflite(keras_pred, tfl_pred))
print('int8  TFLite parity :', compare_keras_vs_tflite(keras_pred, tfl_int8_pred))

fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for i, name in enumerate(['baseline', 'stress', 'amusement']):
    axes[i].scatter(keras_pred[:, i], tfl_pred[:, i], s=10, label='float')
    axes[i].scatter(keras_pred[:, i], tfl_int8_pred[:, i], s=10, alpha=0.5, label='int8')
    axes[i].plot([0, 1], [0, 1], 'k--', alpha=0.4)
    axes[i].set_title(f'P({name})'); axes[i].set_xlabel('Keras'); axes[i].set_ylabel('TFLite')
    axes[i].legend(fontsize=8)
fig.tight_layout()

Artifacts saved under `ml/models/`:
* `mindwave_stress.keras` — full Keras model
* `mindwave_stress.tflite` — float32 TFLite (mobile)
* `mindwave_stress_int8.tflite` — int8 TFLite (smartwatch)
* `scaler.pkl`, `label_map.json`, `feature_names.json`
* `loso_metrics.json`, `holdout_metrics.json`